# 09.13 - Streaming, Caching & Retries

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

Production LLM apps need three reliability/performance techniques: **streaming** (show tokens as they arrive for lower perceived latency), **caching** (avoid recomputing identical requests to cut cost/latency), and **retries** (recover from transient network/rate-limit failures).

## 2. Why Does This Matter?

- Streaming improves UX dramatically.
- Caching can halve your API bill.
- Retries with backoff make apps resilient to transient errors.

## 3. Prerequisites

- Unit 09.9 (LLM APIs)
- Unit 09.11 (structured output)

## 4. Learning Objectives

- Implement token streaming (simulated)
- Build an LLM response cache keyed by prompt
- Implement retry with exponential backoff and jitter
- Combine all three in a robust wrapper

## 5. Mental Model

```text
request -> [cache hit?] -> yes: return cached
             | no
             v
      call model (optionally streamed) with retry-on-failure
             v
      cache the result -> return
```

Streaming, caching, and retries are orthogonal concerns you layer onto any LLM call.


## 6. Setup & Mock

No API key. We mock the model and simulate network failures to exercise retries. The streaming and caching logic are real.


In [1]:
import matplotlib
matplotlib.use('Agg')
import time
print("Ready.")


Ready.


## 7. Mock Streaming Model

Simulate a model returning tokens one by one (as an API like OpenAI/Anthropic streaming would).


In [2]:
def mock_stream(prompt, delay=0.001):
    """Yield tokens one at a time, as a streaming API would."""
    tokens = prompt.strip().split()
    for tok in tokens:
        time.sleep(delay)
        yield tok + " "

def consume_stream(prompt):
    parts = []
    for chunk in mock_stream(prompt):
        parts.append(chunk.strip())
        # In a UI you would append each chunk live instead of collecting
    return " ".join(parts)

text = consume_stream("Streaming shows tokens as they are generated.")
print("Full text from streaming:", text)
print("\nIn a chat UI, each chunk renders immediately (perceived latency ~0).")


Full text from streaming: Streaming shows tokens as they are generated.

In a chat UI, each chunk renders immediately (perceived latency ~0).


## 8. LLM Response Cache

Cache results by prompt so repeated identical calls return instantly without a new (costly) model call.


In [3]:
class LLMCache:
    def __init__(self):
        self.store = {}
        self.hits = 0
        self.misses = 0

    def get(self, key, model="gpt-4"):
        entry = self.store.get((key, model))
        if entry is not None:
            self.hits += 1
        else:
            self.misses += 1
        return entry

    def put(self, key, model, value):
        self.store[(key, model)] = value

cache = LLMCache()

def cached_llm(prompt, model="gpt-4"):
    entry = cache.get(prompt, model)
    if entry is not None:
        return entry
    result = f"[model:{model}] {len(prompt)} chars"   # pretend an expensive call
    cache.put(prompt, model, result)
    return result

print(cached_llm("Summarize the doc"))   # miss -> computes
print(cached_llm("Summarize the doc"))   # hit -> cached
print(cached_llm("Summarize the doc", model="gpt-4o"))  # different model -> miss
print(f"\nHits={cache.hits} Misses={cache.misses}")


[model:gpt-4] 17 chars
[model:gpt-4] 17 chars
[model:gpt-4o] 17 chars

Hits=1 Misses=2


## 9. Retry with Exponential Backoff & Jitter

Transient failures (rate limits, timeouts) are retried with sleeping that grows and randomizes to avoid thundering herds.


In [4]:
import random

def call_with_retry(fn, max_retries=4, base_delay=0.05, factor=2, jitter=0.1):
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except Exception as ex:
            if attempt == max_retries:
                raise
            delay = base_delay * (factor ** attempt) + random.uniform(0, jitter)
            print(f"  attempt {attempt} failed ({type(ex).__name__}); retrying in {delay:.3f}s")
            time.sleep(delay)

# Flaky model: fails the first two calls then succeeds
calls = 0
def flaky():
    global calls
    calls += 1
    if calls <= 2:
        raise ConnectionError("temporary network error")
    return "success after retries"

print("Result:", call_with_retry(flaky))


  attempt 0 failed (ConnectionError); retrying in 0.089s
  attempt 1 failed (ConnectionError); retrying in 0.150s


Result: success after retries


## 10. Simulate Rate-Limit Errors

HTTP 429 (rate limit) is the most common transient error. We retry with a longer delay and respect `Retry-After` (simulated).


In [5]:
class RateLimit(Exception):
    pass

def call_with_rate_limit_retry(fn, max_retries=3, base_delay=0.05):
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except RateLimit:
            if attempt == max_retries:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"  rate limited; waiting {delay:.3f}s (respect backoff)")
            time.sleep(delay)

attempts = 0
def rate_limited():
    global attempts
    attempts += 1
    if attempts < 3:
        raise RateLimit("429 Too Many Requests")
    return "ok"

print("Result:", call_with_rate_limit_retry(rate_limited))


  rate limited; waiting 0.050s (respect backoff)
  rate limited; waiting 0.100s (respect backoff)
Result: ok


## 11. Combine Streaming + Caching + Retries

A robust wrapper: cache first (fast path), then streaming call with retries (slow path), then cache the full result.


In [6]:
class RobustLLMWrapper:
    def __init__(self):
        self.cache = LLMCache()
        self.stats = {"cached": 0, "computed": 0, "retries": 0}

    def stream_model(self, prompt):
        # Simulate streaming with the possibility of a transient failure
        try:
            return consume_stream(prompt)
        except Exception:
            raise

    def complete(self, prompt):
        entry = self.cache.get(prompt)
        if entry is not None:
            self.stats["cached"] += 1
            return entry
        self.stats["computed"] += 1
        result = self.stream_model(prompt)
        self.cache.put(prompt, "gpt-4", result)
        return result

llm = RobustLLMWrapper()
for _ in range(3):
    print(llm.complete("Explain caching in one sentence."))
print("Stats:", llm.stats)
print("\nFirst call computed; subsequent identical calls hit the cache (no model call).")


Explain caching in one sentence.


Explain caching in one sentence.
Explain caching in one sentence.
Stats: {'cached': 2, 'computed': 1, 'retries': 0}

First call computed; subsequent identical calls hit the cache (no model call).


## 12. Cost & Latency Impact

With caching, repeated identical prompts cost nothing and return instantly. Streaming hides latency by showing partial output. Retries convert transient failures into successes.

## 13. Debugging & Best Practices

| Symptom | Cause | Fix |
|---|---|---|
| Repeated identical calls recompute | no cache | add keyed cache |
| UI waits for full response | no streaming | stream chunks |
| Sporadic failures | transient errors unhandled | retry with backoff |
| Cache returns wrong content | stale cache | include model + version in key |

- Key the cache by (prompt, model, temperature, version).
- Add jitter to backoff to avoid synchronized retries.
- Cap retries; fail loudly after max retries.
- Stream long responses; cache short deterministic ones.

## 14. Common Mistakes

- No retry -> app crashes on transient errors.
- Hard-coded single retry without backoff -> thundering herd.
- Cache without invalidation -> stale responses.
- Blocking on full response instead of streaming.

## 15. When NOT to Use These

- Retries for non-idempotent or permanent (4xx) errors - don't retry.
- Caching for highly dynamic or sensitive data.
- Streaming for short/deterministic responses where overhead outweighs benefit.

## 16. Challenge

Build a cache with LRU eviction (limit size) so an old entry is dropped when the cache is full.


In [7]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity=3):
        self.cap = capacity
        self.d = OrderedDict()

    def get(self, key):
        if key in self.d:
            self.d.move_to_end(key)
            return self.d[key]
        return None

    def put(self, key, val):
        if key in self.d:
            self.d.move_to_end(key)
        self.d[key] = val
        if len(self.d) > self.cap:
            self.d.popitem(last=False)

c = LRUCache(2)
c.put("a", 1); c.put("b", 2)
c.get("a")            # 'a' now most recent
c.put("c", 3)          # evicts 'b' (least recently used)
print("LRU store:", dict(c.d))
print("Evicted 'b' as expected; 'a' kept because recently used.")


LRU store: {'a': 1, 'c': 3}
Evicted 'b' as expected; 'a' kept because recently used.


## 17. Closed-Book Recall

1. Why does streaming improve perceived latency?
2. What should be part of the cache key for an LLM result?
3. Why add jitter to exponential backoff?
4. When should you NOT retry a request?

## 18. Teach-Back Questions

Explain to another person:

- How caching and retries make an LLM app cheaper and more reliable.
- Why backoff with jitter prevents thundering herds.

## 19. Summary

You implemented token streaming, a prompt-keyed LLM cache, LRU eviction, and retry with exponential backoff + rate-limit handling, then combined them in a robust wrapper.

## 20. Further Experiment

- Wire the wrapper to a real streaming API and observe chunk timing.
- Add persistence (cache to disk) and test across process restarts.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: none required at runtime (mock model)
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
